# Autoregressive Trajectory Rollout

Generates long protein backbone trajectories by alternating between:
- **Conditional steps**: predict the next frame conditioned on the previous one
- **SDEdit refinement steps**: forward-noise the conditional output to `t_start`, then reverse-denoise with the unconditional model to remove artifacts

**Models are loaded from pre-trained checkpoints in the `upscaled/` directory on Drive.**

**Step pattern** (0-indexed, starting from source frame):
```
Frame 1 → conditional(source)
Frame 2 → sdedit(frame 1)      ← artifact removal
Frame 3 → conditional(frame 2)
Frame 4 → sdedit(frame 3)
...
```

Output: `ca_trajectory.npy` — shape `[num_frames + 1, N, 3]` in Angstroms (frame 0 = source).

## Step 1: Environment Setup

In [ ]:
# Check environment
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Install dependencies
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio'], check=True)
    subprocess.run(['pip', 'install', '-q',
                    'omegaconf', 'pandas', 'tqdm', 'numpy', 'scipy', 'matplotlib',
                    'lightning', 'mdtraj', 'requests', 'einops', 'py3Dmol'], check=True)
else:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'einops', 'py3Dmol'], check=True)

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Ensure the Drive data directory exists (symlink created after repo clone)
    os.makedirs('/content/drive/MyDrive/protein_data/data', exist_ok=True)
    print("Drive mounted")
else:
    os.makedirs('data', exist_ok=True)
    print(f"Local mode; data dir: {os.path.abspath('data')}")

## Step 2: Get Code from GitHub

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/JiwonJJeong/winter-gen-pproject.git"

if IN_COLAB:
    if not os.path.exists('winter-gen-pproject'):
        print(f"Cloning {REPO_URL} ...")
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('winter-gen-pproject')
    # Create data/ symlink inside repo root (after chdir) so all cells find it
    if not os.path.islink('data'):
        os.symlink('/content/drive/MyDrive/protein_data/data', 'data')
        print("data/ -> /content/drive/MyDrive/protein_data/data")
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    for cmd in [
        'find gen_model -name "*.pyc" -delete',
        'find gen_model -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null; true',
    ]:
        subprocess.run(cmd, shell=True)
    print("Code updated")
else:
    if not os.path.isdir('gen_model'):
        raise RuntimeError(
            "gen_model/ not found. Run this notebook from the repository root:\n"
            "  cd /path/to/winter-gen-pproject && jupyter notebook"
        )
    print(f"Local repo root: {os.path.abspath('.')}")

import sys
sys.path.insert(0, '.')
print("Code ready")

## Step 3: Configure Rollout

**Customize these settings before running:**
- `protein.name` — protein identifier matching `atlas.csv`
- `source.frame_idx` — which frame from the trajectory to start from
- `rollout.*` — number of generated frames, SDE steps, SDEdit strength
- `upscaled_dir` — where pre-trained checkpoints live (auto-set from Drive path)

In [ ]:
from omegaconf import OmegaConf

# ── Paths ─────────────────────────────────────────────────────────────────────
# upscaled_dir: directory containing pre-trained conditional + unconditional checkpoints
#   Colab: /content/drive/MyDrive/protein_data/upscaled
#   Local: upscaled/
UPSCALED_DIR = (
    '/content/drive/MyDrive/protein_data/upscaled' if IN_COLAB
    else 'upscaled'
)


# IGSO3 cache: SO3 score lookup table. Storing on Drive avoids ~10 min recompute per session.
IGSO3_CACHE = '/content/drive/MyDrive/protein_data/igso3_cache' if IN_COLAB else '/tmp/igso3_cache'
rollout_config = OmegaConf.create({
    'protein': {
        'name':    '4o66_C',
        'replica': 1,         # trajectory replica to draw the source frame from
    },
    'source': {
        'frame_idx': 1000,    # source frame index in the trajectory
    },
    'rollout': {
        'num_frames':  40,    # total frames to generate (C + R alternating)
        'num_steps':   100,   # reverse-SDE steps per frame (more = better, slower)
        'k':           1,     # temporal gap for conditional steps (1 = next frame)
        't_start':     0.5,   # SDEdit noise level (0–1; higher = more aggressive)
        'noise_scale': 1.0,   # stochasticity in reverse SDE (0 = deterministic)
    },
})

cfg      = rollout_config
prot_cfg = cfg.protein
PROTEIN_FULL_NAME = f"{prot_cfg.name}_R{prot_cfg.replica}"

print("Rollout Configuration:")
print("=" * 60)
print(f"Protein      : {prot_cfg.name}  (replica {prot_cfg.replica})")
print(f"Source frame : {cfg.source.frame_idx}")
print(f"Num frames   : {cfg.rollout.num_frames}  ({cfg.rollout.num_frames // 2} conditional + {cfg.rollout.num_frames // 2} SDEdit)")
print(f"SDE steps    : {cfg.rollout.num_steps} per frame")
print(f"k            : {cfg.rollout.k}  (temporal gap)")
print(f"t_start      : {cfg.rollout.t_start}  (SDEdit noise level)")
print(f"noise_scale  : {cfg.rollout.noise_scale}")
print(f"Upscaled dir : {UPSCALED_DIR}")
print("=" * 60)

## Step 4: Load Protein Data

In [ ]:
import os

if IN_COLAB and not os.path.islink('data'):
    raise RuntimeError(
        "data/ is not linked to Google Drive.\n"
        "Run the 'Mount Google Drive' cell above first, then re-run this cell."
    )

_name = prot_cfg.name
!python scripts/download_and_prep.py {prot_name} --data_dir ./data --out_dir ./data --suffix _latent

traj_path = f'data/{prot_cfg.name}/{PROTEIN_FULL_NAME}_latent.npy'

if os.path.exists(traj_path):
    print(f"\n✅ Trajectory ready : {traj_path}")
else:
    print(f"\n❌ Not found        : {traj_path}")

## Step 5: Visualize Reference MD Frames

Sample frames from the raw MD trajectory and show them before rollout.
The highlighted source frame (red) is the starting point for generation.

In [ ]:
import numpy as np
import pandas as pd
import os
from gen_model.utils.structure_utils import (
    save_ca_trajectory_pdb,
    show_py3dmol_trajectory,
    show_py3dmol_overlay,
)

# Load sequence
_seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
seqres  = _seq_df.loc[prot_cfg.name, 'seqres']
print(f"Protein : {prot_cfg.name}  ({len(seqres)} residues)")

# Sample 30 evenly-spaced Cα frames from the raw MD trajectory
N_REF  = 30
_arr   = np.lib.format.open_memmap(traj_path, 'r')
_idxs  = np.linspace(0, len(_arr) - 1, N_REF, dtype=int)
ref_ca = _arr[_idxs, :, 1, :].astype(np.float32)           # [N_REF, N, 3]
ref_ca = ref_ca - ref_ca.mean(axis=(0, 1), keepdims=True)   # centre at origin

# Source frame (centred)
src_ca_raw = _arr[cfg.source.frame_idx, :, 1, :].astype(np.float32)  # [N, 3]
src_ca     = src_ca_raw - src_ca_raw.mean(axis=0, keepdims=True)     # [N, 3] centred

os.makedirs('outputs', exist_ok=True)
ref_pdb = f'outputs/{prot_cfg.name}_reference.pdb'
save_ca_trajectory_pdb(ref_ca, ref_pdb, seqres)
print(f"Reference PDB saved : {ref_pdb}  ({N_REF} frames)")

try:
    view = show_py3dmol_trajectory(ref_ca, seqres, color='blue', interval=150)
    print(f"\n▶ Cα trace: {N_REF} reference MD frames (blue), animated:")
    view.show()
except ImportError:
    print("py3Dmol not found — install with:  pip install py3Dmol")
    print(f"Or open {ref_pdb} in PyMOL / VMD / UCSF ChimeraX")

In [ ]:
# Highlight source frame in red
try:
    view_src = show_py3dmol_overlay(
        [src_ca[None]],
        seqres,
        colors=['red'],
    )
    print(f"▶ Source frame (frame {cfg.source.frame_idx}, red):")
    view_src.show()
except ImportError:
    pass

## Step 6: Load Models from Upscaled

Discovers and loads the best checkpoint for each model type from `upscaled_dir`.

Expected layout inside `upscaled_dir/`:
```
upscaled/
  conditional_se3/        ← ConditionalSE3Module checkpoints
      best.ckpt  (or any *.ckpt)
  4o66_C_se3/             ← SE3Module (unconditional) checkpoints
      best.ckpt  (or any *.ckpt)
```

In [ ]:
import glob
import torch

from gen_model.train_base             import default_se3_conf, default_model_conf
from gen_model.train_conditional      import ConditionalSE3Module
from gen_model.train_unconditional    import SE3Module
from gen_model.diffusion.se3_diffuser import SE3Diffuser
from gen_model.inference_conditional  import compute_coord_scale

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")


def _find_best_ckpt(dirpath: str) -> str:
    """Return the best non-last checkpoint in dirpath, or raise if none found."""
    all_ckpts = sorted(glob.glob(os.path.join(dirpath, '*.ckpt')))
    ranked    = [p for p in all_ckpts if 'last' not in os.path.basename(p)]
    # Prefer explicit 'best.ckpt', then ranked-by-name (val_loss in filename)
    best_explicit = [p for p in ranked if 'best' in os.path.basename(p)]
    if best_explicit:
        return best_explicit[-1]
    if ranked:
        return ranked[-1]
    if all_ckpts:
        return all_ckpts[-1]  # fall back to last.ckpt
    raise FileNotFoundError(f"No .ckpt files found in {dirpath}")


# ── Locate checkpoints ────────────────────────────────────────────────────────
COND_CKPT_DIR   = os.path.join(UPSCALED_DIR, 'conditional_se3')
UNCOND_CKPT_DIR = os.path.join(UPSCALED_DIR, f'{prot_cfg.name}_se3')

cond_ckpt   = _find_best_ckpt(COND_CKPT_DIR)
uncond_ckpt = _find_best_ckpt(UNCOND_CKPT_DIR)

print(f"Conditional   : {cond_ckpt}")
print(f"Unconditional : {uncond_ckpt}")

# ── Shared configs ────────────────────────────────────────────────────────────
se3_conf    = default_se3_conf(cache_dir=IGSO3_CACHE)
cond_conf   = default_model_conf(use_temporal_embedding=True)
uncond_conf = default_model_conf(use_temporal_embedding=False)

# ── Load models ───────────────────────────────────────────────────────────────
conditional_model = ConditionalSE3Module.load_from_checkpoint(
    cond_ckpt,
    model_conf=cond_conf,
    se3_conf=se3_conf,
    map_location=device,
).to(device).eval()

unconditional_model = SE3Module.load_from_checkpoint(
    uncond_ckpt,
    model_conf=uncond_conf,
    se3_conf=se3_conf,
    map_location=device,
).to(device).eval()

print("\n✅ Both models loaded and ready")

# ── Diffuser + coord_scale ────────────────────────────────────────────────────
diffuser     = SE3Diffuser(se3_conf)
coord_scale  = compute_coord_scale(traj_path)
print(f"coord_scale  : {coord_scale:.4f}  (computed from trajectory)")

## Step 7: Autoregressive Rollout

Alternates **conditional** (predict next frame) and **SDEdit refinement** (remove artifacts) steps.

```
source → [C] → [R] → [C] → [R] → ...
```

Each conditional step runs a full reverse SE(3) SDE (`num_steps` denoising steps).
Each refinement step noises to `t_start` then denoises back to t=0 with the unconditional model.

In [ ]:
import torch
import numpy as np
from gen_model.inference_conditional import load_source_frame
from gen_model.video_extrapolation   import build_trajectory
from gen_model.utils.structure_utils import rigids_to_ca_trajectory

# ── Source frame ─────────────────────────────────────────────────────────────
atom14, seqres, ca_pos, rigids_0, aatype = load_source_frame(
    traj_path, cfg.source.frame_idx, 'gen_model/splits/atlas.csv', prot_cfg.name
)
N        = len(seqres)
centroid = ca_pos.mean(axis=0, keepdims=True)                           # [1, 3]
sc_ca_t  = torch.from_numpy((ca_pos - centroid) * coord_scale).float() # [N, 3]
res_mask = torch.ones(N, dtype=torch.float32)

print(f"Protein     : {prot_cfg.name}  ({N} residues)")
print(f"Source frame: index {cfg.source.frame_idx}")
print(f"coord_scale : {coord_scale:.4f}")
print()
print(f"Generating {cfg.rollout.num_frames} frames  "
      f"({cfg.rollout.num_steps} SDE steps each)  on {device} ...")
print()

# ── Run rollout ───────────────────────────────────────────────────────────────
frames = build_trajectory(
    conditional_model   = conditional_model,
    unconditional_model = unconditional_model,
    diffuser            = diffuser,
    sc_ca_t             = sc_ca_t,
    aatype              = aatype,
    res_mask            = res_mask,
    k                   = cfg.rollout.k,
    num_frames          = cfg.rollout.num_frames,
    num_steps           = cfg.rollout.num_steps,
    t_start             = cfg.rollout.t_start,
    device              = device,
    noise_scale         = cfg.rollout.noise_scale,
)

# ── Convert to Angstroms and prepend source frame ─────────────────────────────
generated_ca = rigids_to_ca_trajectory(frames, coord_scale)  # [num_frames, N, 3]
source_ca    = (ca_pos - centroid).astype(np.float32)        # [N, 3] centred
trajectory   = np.concatenate(
    [source_ca[None], generated_ca], axis=0
)                                                            # [num_frames+1, N, 3]

print(f"\n✅ Rollout complete")
print(f"   Trajectory shape : {trajectory.shape}  (frames × residues × xyz, Å)")
print(f"   Frame 0          : source (index {cfg.source.frame_idx})")
print(f"   Frames 1–{cfg.rollout.num_frames}     : generated")

## Step 8: Visualize Generated Trajectory

In [ ]:
# Generated trajectory (orange, animated)
try:
    view_gen = show_py3dmol_trajectory(
        trajectory, seqres, color='orange', interval=200,
    )
    print(f"▶ Generated trajectory ({trajectory.shape[0]} frames, orange, animated):")
    view_gen.show()
except ImportError:
    print("py3Dmol not installed — install with:  pip install py3Dmol")

In [ ]:
# Overlay: source frame (blue) vs final generated frame (orange)
try:
    view_cmp = show_py3dmol_overlay(
        [source_ca[None], generated_ca[[-1]]],
        seqres,
        colors=['blue', 'orange'],
    )
    print("▶ Source (blue) vs final generated frame (orange):")
    view_cmp.show()
except ImportError:
    pass

In [ ]:
# Overlay: 5 evenly-spaced frames through the generated trajectory
try:
    import numpy as np
    _N_OVERLAY = 5
    _idxs      = np.linspace(0, len(trajectory) - 1, _N_OVERLAY, dtype=int)
    _frames    = [trajectory[[i]] for i in _idxs]
    _colors    = ['blue', 'cyan', 'green', 'orange', 'red']
    view_evo   = show_py3dmol_overlay(_frames, seqres, colors=_colors)
    print(f"▶ Structural evolution — {_N_OVERLAY} evenly-spaced frames (blue→red):")
    view_evo.show()
except ImportError:
    pass

## Step 9: Quantitative Analysis

- **Cα RMSD from source** over time — measures structural drift along the trajectory
- **Pairwise RMSD heatmap** — shows how diverse the generated frames are

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Cα RMSD from source over time ─────────────────────────────────────────────
def _rmsd(a, b):
    """Cα RMSD between two [N, 3] arrays (no alignment)."""
    return float(np.sqrt(np.mean(np.sum((a - b) ** 2, axis=-1))))

rmsds = np.array([_rmsd(trajectory[i], source_ca) for i in range(len(trajectory))])

# ── Pairwise RMSD heatmap ──────────────────────────────────────────────────────
T = len(trajectory)
pairwise = np.zeros((T, T), dtype=np.float32)
for i in range(T):
    for j in range(i, T):
        r = _rmsd(trajectory[i], trajectory[j])
        pairwise[i, j] = r
        pairwise[j, i] = r

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: RMSD from source
step_pattern = ['source'] + [
    'C' if (i % 2 == 0) else 'R'
    for i in range(cfg.rollout.num_frames)
]
colors_per_step = ['gray'] + [
    'steelblue' if s == 'C' else 'coral'
    for s in step_pattern[1:]
]
axes[0].bar(range(T), rmsds, color=colors_per_step, alpha=0.8, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Frame index')
axes[0].set_ylabel('Cα RMSD from source (Å)')
axes[0].set_title(f'Drift from source — {prot_cfg.name}')
axes[0].grid(alpha=0.3, axis='y')
# Legend
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(facecolor='steelblue', label='Conditional (C)'),
    Patch(facecolor='coral',     label='SDEdit (R)'),
], loc='upper left')

# Right: pairwise RMSD heatmap
im = axes[1].imshow(pairwise, cmap='viridis', aspect='auto')
axes[1].set_xlabel('Frame index')
axes[1].set_ylabel('Frame index')
axes[1].set_title('Pairwise Cα RMSD (Å)')
plt.colorbar(im, ax=axes[1], label='RMSD (Å)')

plt.tight_layout()
plt.savefig(f'outputs/{prot_cfg.name}_rollout_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Mean RMSD from source : {rmsds[1:].mean():.2f} Å")
print(f"Max  RMSD from source : {rmsds[1:].max():.2f} Å  (frame {rmsds[1:].argmax() + 1})")
print(f"Mean pairwise RMSD    : {pairwise[np.triu_indices(T, k=1)].mean():.2f} Å")

## Step 10: Save Results and Download

In [ ]:
import os
import numpy as np
from gen_model.utils.structure_utils import save_ca_trajectory_pdb

out_dir = f'outputs/{prot_cfg.name}_rollout'
os.makedirs(out_dir, exist_ok=True)

# Save trajectory as .npy  [T, N, 3]  (T = num_frames + 1, frame 0 = source)
npy_path = os.path.join(out_dir, 'ca_trajectory.npy')
np.save(npy_path, trajectory)
print(f"✅ CA trajectory saved : {npy_path}  shape={trajectory.shape}")

# Save as multi-model PDB (opens directly in PyMOL / VMD / ChimeraX)
pdb_path = os.path.join(out_dir, 'ca_trajectory.pdb')
save_ca_trajectory_pdb(trajectory, pdb_path, seqres)
print(f"✅ PDB trajectory saved: {pdb_path}")

# Save analysis plot
import shutil
plot_src = f'outputs/{prot_cfg.name}_rollout_analysis.png'
if os.path.exists(plot_src):
    shutil.copy(plot_src, out_dir)
    print(f"✅ Analysis plot saved : {out_dir}/{os.path.basename(plot_src)}")

# Metadata
import json
meta = {
    'protein':          prot_cfg.name,
    'replica':          prot_cfg.replica,
    'source_frame_idx': cfg.source.frame_idx,
    'num_frames':       cfg.rollout.num_frames,
    'num_steps':        cfg.rollout.num_steps,
    'k':                cfg.rollout.k,
    't_start':          cfg.rollout.t_start,
    'noise_scale':      cfg.rollout.noise_scale,
    'coord_scale':      float(coord_scale),
    'conditional_ckpt': cond_ckpt,
    'unconditional_ckpt': uncond_ckpt,
    'trajectory_shape': list(trajectory.shape),
}
meta_path = os.path.join(out_dir, 'rollout_config.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f"✅ Metadata saved      : {meta_path}")

In [ ]:
import os, subprocess

save_zip = f'{prot_cfg.name}_rollout_results.zip'
dirs_to_pack = [d for d in [out_dir] if os.path.isdir(d)]

if dirs_to_pack:
    subprocess.run(['zip', '-rq', save_zip] + dirs_to_pack, check=True)
    print(f"✅ Packaged: {save_zip}  ({', '.join(dirs_to_pack)})")
    if IN_COLAB:
        from google.colab import files
        files.download(save_zip)
        print("✅ Download started")
    else:
        print(f"✅ Saved locally as {save_zip}")
else:
    print("Nothing to package yet — run Steps 7 and 10 first.")

## Summary

**What was done:**
1. ✓ Loaded pre-trained conditional + unconditional models from `upscaled/`
2. ✓ Started from source frame `{cfg.source.frame_idx}` of `{prot_cfg.name}_R{prot_cfg.replica}`
3. ✓ Generated `{cfg.rollout.num_frames}` frames autoregressively (C→R→C→R...)
4. ✓ Visualized trajectory and structural drift
5. ✓ Saved `ca_trajectory.npy`, `ca_trajectory.pdb`, and analysis plots

**To run from the terminal instead:**
```bash
python gen_model/video_extrapolation.py \
    --conditional_ckpt   upscaled/conditional_se3/best.ckpt \
    --unconditional_ckpt upscaled/4o66_C_se3/best.ckpt \
    --npy_path           data/4o66_C/4o66_C_R1_latent.npy \
    --protein_name       4o66_C \
    --frame_idx          1000 \
    --num_frames         40 \
    --num_steps          100 \
    --k                  1 \
    --t_start            0.5 \
    --out_dir            outputs/4o66_C_rollout
```

**To generate a different trajectory:**
- Change `cfg.source.frame_idx` and re-run from Step 7
- Adjust `cfg.rollout.num_frames`, `k`, or `t_start` for different rollout behaviours